# Extended Lead-Time Geomagnetic Storm Forecasting With Solar Wind Ensembles and Machine Learning — Implementation / 구현

**Paper**: Billcliff, M., Smith, A. W., Owens, M., Woo, W. L., Barnard, L., Edward-Inatimi, N., & Rae, I. J. (2026). *Space Weather*, 24, e2025SW004823.

이 노트북은 논문의 핵심 알고리즘을 구현합니다:
This notebook implements the key algorithms from the paper:

1. **Part 1**: Carrington map 위 앙상블 섭동 경로 생성 / Ensemble perturbation path generation on Carrington maps
2. **Part 2**: 합성 태양풍 데이터로 1D HUXt-style 전파 시뮬레이션 / Simplified 1D HUXt-style propagation with synthetic solar wind data
3. **Part 3**: 로지스틱 회귀 분류기 (NumPy 직접 구현) / Logistic regression classifier from scratch (NumPy only)
4. **Part 4**: MAE 가중 앙상블 집약 / MAE-weighted ensemble aggregation
5. **Part 5**: 전체 파이프라인 통합 (합성 데이터) / Full pipeline integration with synthetic data
6. **Part 6**: 평가 메트릭 — ROC AUC & BSS / Evaluation metrics — ROC AUC & BSS
7. **Part 7**: scikit-learn 비교 / Comparison with scikit-learn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

np.random.seed(42)

## Part 1: Ensemble Perturbation Path Generation / 앙상블 섭동 경로 생성

논문의 핵심 아이디어: MAS Carrington map 위에서 지구의 경로를 사인파로 섭동하여
다양한 태양풍 속도 프로파일 앙상블을 추출합니다.

The paper's key idea: sinusoidal perturbation of Earth's path on MAS Carrington maps
to extract diverse solar wind velocity profile ensembles.

$$\theta(\phi) = \theta_E + \theta_{MAX} \sin(\phi + \phi_0)$$

- $\theta_{MAX} \sim N(0, \sigma^2)$, $\sigma = 7.5°$
- $\phi_0 \sim U(0, 2\pi)$

먼저 합성 Carrington map을 생성하고, 위에서 섭동 경로를 시각화합니다.
First we create a synthetic Carrington map, then visualize perturbation paths on it.

In [ ]:
def create_synthetic_carrington_map(
    n_lon: int = 360,
    n_lat: int = 180,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Create a synthetic Carrington map mimicking solar wind velocity structure.

    Simulates the bimodal solar wind: slow wind (~350 km/s) near the
    equatorial streamer belt and fast wind (~600 km/s) from coronal holes
    at higher latitudes, with longitudinal structure from CIRs.

    Args:
        n_lon: Number of longitude grid points.
        n_lat: Number of latitude grid points.

    Returns:
        Tuple of (longitude_grid, latitude_grid, velocity_map) arrays.
    """
    lon = np.linspace(0, 360, n_lon)
    lat = np.linspace(-90, 90, n_lat)
    lon_grid, lat_grid = np.meshgrid(lon, lat)

    # Base: slow wind near equator, fast wind at poles
    v_base = 350 + 250 * (np.abs(lat_grid) / 90) ** 1.5

    # Streamer belt: narrow slow-wind band oscillating in latitude
    streamer_lat = 10 * np.sin(np.radians(lon_grid) * 2)
    streamer_mask = np.exp(-0.5 * ((lat_grid - streamer_lat) / 15) ** 2)
    v_base -= 100 * streamer_mask

    # Coronal holes: localized fast-wind regions
    for ch_lon, ch_lat, ch_size in [(120, 30, 20), (250, -25, 25), (50, -40, 15)]:
        ch_mask = np.exp(
            -0.5 * (((lon_grid - ch_lon) / ch_size) ** 2 + ((lat_grid - ch_lat) / ch_size) ** 2)
        )
        v_base += 150 * ch_mask

    v_base = np.clip(v_base, 250, 700)
    return lon_grid, lat_grid, v_base


def generate_perturbation_paths(
    n_ensemble: int = 100,
    theta_e: float = 0.0,
    sigma: float = 7.5,
    n_lon: int = 360,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Generate sinusoidal perturbation paths on a Carrington map.

    Implements Equation 1 from the paper:
        theta(phi) = theta_E + theta_MAX * sin(phi + phi_0)

    Args:
        n_ensemble: Number of ensemble members.
        theta_e: Unperturbed Earth heliolatitude in degrees.
        sigma: Standard deviation of theta_MAX distribution in degrees.
        n_lon: Number of longitude grid points.

    Returns:
        Tuple of (longitudes, paths array [n_ensemble x n_lon],
                  (theta_max_values, phi_0_values)).
    """
    lon = np.linspace(0, 360, n_lon)
    lon_rad = np.radians(lon)

    theta_max = np.random.normal(0, sigma, n_ensemble)
    phi_0 = np.random.uniform(0, 2 * np.pi, n_ensemble)

    paths = np.zeros((n_ensemble, n_lon))
    for k in range(n_ensemble):
        paths[k] = theta_e + theta_max[k] * np.sin(lon_rad + phi_0[k])

    return lon, paths, (theta_max, phi_0)


def extract_velocity_profiles(
    lon_grid: np.ndarray,
    lat_grid: np.ndarray,
    v_map: np.ndarray,
    lon: np.ndarray,
    paths: np.ndarray,
) -> np.ndarray:
    """Extract velocity profiles along perturbation paths from a Carrington map.

    Uses nearest-neighbor interpolation to sample velocity values
    along each ensemble member's perturbed path.

    Args:
        lon_grid: 2D longitude grid from Carrington map.
        lat_grid: 2D latitude grid from Carrington map.
        v_map: 2D velocity map.
        lon: 1D longitude array for paths.
        paths: 2D array of perturbation paths [n_ensemble x n_lon].

    Returns:
        Velocity profiles array [n_ensemble x n_lon].
    """
    lat_vals = lat_grid[:, 0]
    n_ensemble = paths.shape[0]
    v_profiles = np.zeros_like(paths)

    for k in range(n_ensemble):
        for i, (lo, la) in enumerate(zip(lon, paths[k])):
            lon_idx = np.argmin(np.abs(lon_grid[0, :] - lo))
            lat_idx = np.argmin(np.abs(lat_vals - la))
            v_profiles[k, i] = v_map[lat_idx, lon_idx]

    return v_profiles


# --- Generate synthetic Carrington map and ensemble ---
lon_grid, lat_grid, v_map = create_synthetic_carrington_map()
lon, paths, (theta_max_vals, phi_0_vals) = generate_perturbation_paths(
    n_ensemble=50, sigma=7.5
)
v_profiles = extract_velocity_profiles(lon_grid, lat_grid, v_map, lon, paths)

# --- Visualization (reproducing Figure 3 concept) ---
fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [1.2, 1]})

# Top: Carrington map with perturbation paths
ax = axes[0]
im = ax.pcolormesh(lon_grid, lat_grid, v_map, cmap='inferno', shading='auto')
for k in range(min(50, paths.shape[0])):
    ax.plot(lon, paths[k], 'w-', alpha=0.3, linewidth=0.5)
ax.plot(lon, np.zeros_like(lon), 'c--', linewidth=2, label='Earth (unperturbed)')
ax.set_xlabel('Carrington Longitude (deg)')
ax.set_ylabel('Latitude (deg)')
ax.set_title('Synthetic Carrington Map with Ensemble Perturbation Paths\n'
             '합성 Carrington Map과 앙상블 섭동 경로')
ax.legend(loc='upper right')
plt.colorbar(im, ax=ax, label='Velocity (km/s)')

# Bottom: Extracted velocity profiles
ax = axes[1]
for k in range(v_profiles.shape[0]):
    ax.plot(lon, v_profiles[k], alpha=0.2, color='steelblue', linewidth=0.5)
ax.plot(lon, v_profiles.mean(axis=0), 'r-', linewidth=2, label='Ensemble mean')
ax.set_xlabel('Carrington Longitude (deg)')
ax.set_ylabel('Velocity (km/s)')
ax.set_title('Extracted Solar Wind Velocity Profiles / 추출된 태양풍 속도 프로파일')
ax.legend()

plt.tight_layout()
plt.show()

print(f"Ensemble size: {v_profiles.shape[0]}")
print(f"θ_MAX: mean={theta_max_vals.mean():.2f}°, std={theta_max_vals.std():.2f}°")
print(f"Velocity range: {v_profiles.min():.0f} – {v_profiles.max():.0f} km/s")

## Part 2: Simplified 1D Solar Wind Propagation / 간략화된 1D 태양풍 전파

HUXt는 1D incompressible hydrodynamic 방정식을 풀어 태양풍을 21.5 $R_\odot$에서 1 AU까지 전파합니다.
여기서는 핵심 개념을 보여주기 위해 간단한 kinematic 전파 모델을 구현합니다:
태양풍 속도에 따라 도착 시간이 달라지는 효과를 시뮬레이션합니다.

HUXt solves 1D incompressible hydrodynamic equations to propagate solar wind from 21.5 $R_\odot$ to 1 AU.
Here we implement a simplified kinematic propagation to demonstrate the key concept:
different solar wind speeds lead to different arrival times at Earth.

**전파 시간 / Transit time**: $\Delta t = \frac{r_{Earth} - r_{source}}{v}$

- $r_{Earth} = 1$ AU $= 1.496 \times 10^{11}$ m
- $r_{source} = 21.5 R_\odot = 21.5 \times 6.957 \times 10^{8}$ m $\approx 0.1$ AU

In [ ]:
def kinematic_propagation(
    v_source: np.ndarray,
    dt_source_hours: float = 2.0,
    r_source_au: float = 0.1,
    r_earth_au: float = 1.0,
    dt_output_hours: float = 1.0,
    n_output_hours: int = 720,
) -> Tuple[np.ndarray, np.ndarray]:
    """Simplified kinematic solar wind propagation from source to Earth.

    Each parcel of solar wind travels radially at its initial speed.
    Fast parcels overtake slow ones (stream interaction simplified
    by capping velocity at the preceding fast stream).

    Args:
        v_source: Velocity time series at the source surface (km/s).
        dt_source_hours: Time step of source data in hours.
        r_source_au: Radial distance of source in AU.
        r_earth_au: Radial distance of Earth in AU.
        dt_output_hours: Output time step in hours.
        n_output_hours: Number of output time steps.

    Returns:
        Tuple of (time_hours, velocity_at_earth) arrays.
    """
    au_to_km = 1.496e8  # 1 AU in km
    dr = (r_earth_au - r_source_au) * au_to_km  # distance in km

    # Compute arrival time for each source parcel
    t_output = np.arange(n_output_hours) * dt_output_hours
    v_earth = np.full(n_output_hours, np.nan)

    for i, v in enumerate(v_source):
        t_emit = i * dt_source_hours
        transit_hours = (dr / v) / 3600  # km / (km/s) -> s -> hours
        t_arrive = t_emit + transit_hours

        # Find closest output time bin
        idx = int(round(t_arrive / dt_output_hours))
        if 0 <= idx < n_output_hours:
            # If multiple parcels arrive at same time, take the faster one
            # (simplified stream interaction)
            if np.isnan(v_earth[idx]) or v > v_earth[idx]:
                v_earth[idx] = v

    # Fill gaps with linear interpolation
    valid = ~np.isnan(v_earth)
    if valid.any():
        v_earth = np.interp(t_output, t_output[valid], v_earth[valid])

    return t_output, v_earth


# --- Propagate ensemble velocity profiles ---
n_ensemble = 50
# Convert longitude to time: 360 deg ≈ 27.28 days
lon_to_hours = 27.28 * 24 / 360  # hours per degree
dt_source = lon_to_hours * (lon[1] - lon[0])  # time step between longitude points

t_earth_all = []
v_earth_all = []

for k in range(n_ensemble):
    t_earth, v_earth = kinematic_propagation(
        v_profiles[k], dt_source_hours=dt_source
    )
    t_earth_all.append(t_earth)
    v_earth_all.append(v_earth)

v_earth_arr = np.array(v_earth_all)
t_days = t_earth_all[0] / 24

# Create synthetic "OMNI" observation (ensemble mean + noise)
v_omni = v_earth_arr.mean(axis=0) + np.random.normal(0, 20, v_earth_arr.shape[1])
v_omni = np.clip(v_omni, 250, 700)

# --- Visualization ---
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

ax = axes[0]
for k in range(n_ensemble):
    ax.plot(t_days, v_earth_arr[k], alpha=0.15, color='steelblue', linewidth=0.5)
ax.plot(t_days, v_earth_arr.mean(axis=0), 'r-', linewidth=2, label='Ensemble mean')
ax.plot(t_days, v_omni, 'k-', linewidth=1.5, alpha=0.7, label='Synthetic OMNI')
ax.set_ylabel('Velocity (km/s)')
ax.set_title('Solar Wind Ensemble at Earth (1 AU) / 지구(1 AU)에서의 태양풍 앙상블')
ax.legend()
ax.set_xlim(0, 27)

ax = axes[1]
ax.plot(t_days, v_earth_arr.mean(axis=0) - v_omni, 'b-', alpha=0.7, label='v_ensemble - OMNI')
ax.axhline(0, color='k', linestyle='--', alpha=0.3)
ax.set_xlabel('Time (days)')
ax.set_ylabel('Δv (km/s)')
ax.set_title('Ensemble Mean − OMNI Residual / 앙상블 평균 − OMNI 잔차')
ax.legend()
ax.set_xlim(0, 27)

plt.tight_layout()
plt.show()

# Transit time statistics
transit_300 = ((1.0 - 0.1) * 1.496e8 / 300) / 3600 / 24
transit_600 = ((1.0 - 0.1) * 1.496e8 / 600) / 3600 / 24
print(f"Transit time at 300 km/s: {transit_300:.1f} days")
print(f"Transit time at 600 km/s: {transit_600:.1f} days")
print(f"This is why lead times of 1-3 days are achievable!")

## Part 3: Logistic Regression from Scratch / 로지스틱 회귀 직접 구현

논문에서 각 앙상블 멤버에 대해 개별 로지스틱 회귀 분류기를 훈련합니다 (Equation 2):

$$P(Y=1|X) = \sigma(\mathbf{w}^\top \mathbf{x} + b) = \frac{1}{1 + e^{-(\mathbf{w}^\top \mathbf{x} + b)}}$$

손실 함수 (Binary Cross-Entropy):

$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N}\left[y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)\right]$$

여기서는 gradient descent로 직접 학습합니다.
Here we train using gradient descent from scratch.

In [ ]:
class LogisticRegressionScratch:
    """Logistic regression classifier implemented from scratch using NumPy.

    Implements the Model C(i) component from the paper: a binary classifier
    that outputs storm probability given solar wind features.

    Attributes:
        weights: Learned weight vector.
        bias: Learned bias term.
        losses: Training loss history.
    """

    def __init__(self, learning_rate: float = 0.01, n_iterations: int = 1000):
        """Initialize logistic regression.

        Args:
            learning_rate: Step size for gradient descent.
            n_iterations: Number of training iterations.
        """
        self.lr = learning_rate
        self.n_iter = n_iterations
        self.weights = None
        self.bias = None
        self.losses = []

    @staticmethod
    def _sigmoid(z: np.ndarray) -> np.ndarray:
        """Numerically stable sigmoid function."""
        return np.where(
            z >= 0,
            1 / (1 + np.exp(-z)),
            np.exp(z) / (1 + np.exp(z))
        )

    def fit(self, X: np.ndarray, y: np.ndarray) -> 'LogisticRegressionScratch':
        """Train the model using gradient descent.

        Args:
            X: Feature matrix of shape (n_samples, n_features).
            y: Binary labels of shape (n_samples,).

        Returns:
            Self for method chaining.
        """
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0.0
        self.losses = []

        for _ in range(self.n_iter):
            z = X @ self.weights + self.bias
            y_hat = self._sigmoid(z)

            # Binary cross-entropy loss
            eps = 1e-15
            loss = -np.mean(y * np.log(y_hat + eps) + (1 - y) * np.log(1 - y_hat + eps))
            self.losses.append(loss)

            # Gradients
            dw = (1 / n_samples) * (X.T @ (y_hat - y))
            db = (1 / n_samples) * np.sum(y_hat - y)

            self.weights -= self.lr * dw
            self.bias -= self.lr * db

        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Predict storm probability.

        Args:
            X: Feature matrix of shape (n_samples, n_features).

        Returns:
            Probability array of shape (n_samples,).
        """
        return self._sigmoid(X @ self.weights + self.bias)

    def predict(self, X: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        """Predict binary class.

        Args:
            X: Feature matrix of shape (n_samples, n_features).
            threshold: Decision threshold.

        Returns:
            Binary prediction array of shape (n_samples,).
        """
        return (self.predict_proba(X) >= threshold).astype(int)


# --- Generate synthetic storm forecasting dataset ---
def generate_storm_dataset(
    n_samples: int = 2000,
    n_ensemble: int = 20,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Generate a synthetic storm forecasting dataset.

    Simulates the paper's setup: for each time window, we have ensemble
    solar wind velocities and derived features. Higher velocities and
    larger velocity gradients correlate with storm occurrence.

    Args:
        n_samples: Number of forecast windows.
        n_ensemble: Number of ensemble members.

    Returns:
        Tuple of (features_per_member [n_ensemble, n_samples, n_features],
                  labels [n_samples], mae_per_member [n_ensemble]).
    """
    # Underlying "true" solar wind speed for each window
    v_true = np.random.uniform(300, 650, n_samples)

    # Storm probability increases with velocity (simplified)
    storm_prob = 1 / (1 + np.exp(-0.02 * (v_true - 450)))
    labels = (np.random.rand(n_samples) < storm_prob).astype(int)

    # Balance the dataset (as done in the paper)
    idx_storm = np.where(labels == 1)[0]
    idx_nostorm = np.where(labels == 0)[0]
    n_min = min(len(idx_storm), len(idx_nostorm))
    idx_balanced = np.concatenate([
        np.random.choice(idx_storm, n_min, replace=False),
        np.random.choice(idx_nostorm, n_min, replace=False)
    ])
    np.random.shuffle(idx_balanced)

    v_true = v_true[idx_balanced]
    labels = labels[idx_balanced]
    n_samples = len(labels)

    # Generate ensemble member features
    features = np.zeros((n_ensemble, n_samples, 4))
    mae_vals = np.zeros(n_ensemble)

    for k in range(n_ensemble):
        # Each member has a different bias and noise level
        bias_k = np.random.normal(0, 30)
        noise_k = np.random.uniform(20, 80)

        v_k = v_true + bias_k + np.random.normal(0, noise_k, n_samples)
        dv_k = np.gradient(v_k)
        v_minus_omni = v_k - v_true + np.random.normal(0, 10, n_samples)

        # Synthetic Hp30 in input window (correlated with storm)
        hp30_k = 2 + 3 * labels + np.random.normal(0, 1, n_samples)

        features[k, :, 0] = v_k            # predicted velocity
        features[k, :, 1] = dv_k           # velocity gradient
        features[k, :, 2] = v_minus_omni   # v - OMNI
        features[k, :, 3] = hp30_k         # Hp30

        mae_vals[k] = np.mean(np.abs(v_k - v_true))

    return features, labels, mae_vals


# Generate dataset
features, labels, mae_vals = generate_storm_dataset(n_samples=2000, n_ensemble=20)
n_ensemble_demo = features.shape[0]
n_samples_demo = features.shape[1]

# Train/test split (80/20 as in the paper)
split_idx = int(0.8 * n_samples_demo)
X_trains = features[:, :split_idx, :]
X_tests = features[:, split_idx:, :]
y_train = labels[:split_idx]
y_test = labels[split_idx:]

# Train individual classifiers per ensemble member
classifiers = []
train_losses = []

for k in range(n_ensemble_demo):
    # Standardize features
    mu = X_trains[k].mean(axis=0)
    sigma = X_trains[k].std(axis=0) + 1e-8
    X_tr = (X_trains[k] - mu) / sigma
    X_te = (X_tests[k] - mu) / sigma

    clf = LogisticRegressionScratch(learning_rate=0.1, n_iterations=500)
    clf.fit(X_tr, y_train)
    clf._mu = mu  # store for later use
    clf._sigma = sigma
    classifiers.append(clf)
    train_losses.append(clf.losses)

# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training loss curves
ax = axes[0]
for k in range(n_ensemble_demo):
    ax.plot(train_losses[k], alpha=0.3, linewidth=0.5)
ax.set_xlabel('Iteration')
ax.set_ylabel('Binary Cross-Entropy Loss')
ax.set_title(f'Training Loss for {n_ensemble_demo} Ensemble Classifiers\n'
             f'{n_ensemble_demo}개 앙상블 분류기의 훈련 손실')

# Individual classifier probabilities on test set
ax = axes[1]
probs_all = []
for k in range(n_ensemble_demo):
    X_te = (X_tests[k] - classifiers[k]._mu) / classifiers[k]._sigma
    probs_k = classifiers[k].predict_proba(X_te)
    probs_all.append(probs_k)
    ax.scatter(range(len(probs_k)), probs_k, alpha=0.05, s=1, c='steelblue')

probs_all = np.array(probs_all)
ax.scatter(range(len(y_test)), y_test, alpha=0.3, s=5, c='red', label='True label')
ax.set_xlabel('Test sample index')
ax.set_ylabel('Storm Probability')
ax.set_title('Individual Classifier Predictions on Test Set\n'
             '개별 분류기의 테스트셋 예측')
ax.legend()

plt.tight_layout()
plt.show()

print(f"Dataset: {n_samples_demo} samples ({y_train.sum()} train storms, {y_test.sum()} test storms)")
print(f"Ensemble: {n_ensemble_demo} classifiers trained")

## Part 4: MAE-Weighted Ensemble Aggregation / MAE 가중 앙상블 집약

논문의 Model C(ii): 개별 분류기의 확률 출력을 MAE 가중 평균으로 집약합니다 (Equations 3-5).

Paper's Model C(ii): aggregate individual classifier probability outputs via MAE-weighted mean (Equations 3-5).

$$w_j = \frac{1}{MAE_j^2}, \quad w_j^{norm} = \frac{w_j}{\sum_j w_j}, \quad \hat{y} = \sum_j w_j^{norm} \cdot p_j$$

MAE가 작은(더 정확한) 멤버에 더 큰 가중치를 부여합니다.
Members with smaller MAE (more accurate) receive larger weights.

In [ ]:
def mae_weighted_mean(
    probs: np.ndarray,
    mae_values: np.ndarray,
) -> np.ndarray:
    """Compute MAE-weighted mean of ensemble probabilities (Equations 3-5).

    Args:
        probs: Probability array of shape (n_ensemble, n_samples).
        mae_values: MAE values for each ensemble member (n_ensemble,).

    Returns:
        Weighted probability array of shape (n_samples,).
    """
    # Eq 3: w_j = 1 / MAE_j^2
    w = 1.0 / (mae_values ** 2)

    # Eq 4: normalize weights
    w_norm = w / w.sum()

    # Eq 5: weighted mean
    y_hat = w_norm @ probs  # (n_ensemble,) @ (n_ensemble, n_samples) -> (n_samples,)
    return y_hat


def simple_mean(probs: np.ndarray) -> np.ndarray:
    """Compute unweighted mean of ensemble probabilities.

    Args:
        probs: Probability array of shape (n_ensemble, n_samples).

    Returns:
        Mean probability array of shape (n_samples,).
    """
    return probs.mean(axis=0)


# --- Compute weighted and unweighted ensemble predictions ---
y_pred_weighted = mae_weighted_mean(probs_all, mae_vals)
y_pred_simple = simple_mean(probs_all)

# --- Visualization ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Weight distribution
ax = axes[0]
w = 1.0 / (mae_vals ** 2)
w_norm = w / w.sum()
sorted_idx = np.argsort(mae_vals)
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(mae_vals)))
ax.bar(range(len(mae_vals)), w_norm[sorted_idx], color=colors)
ax.set_xlabel('Ensemble member (sorted by MAE)')
ax.set_ylabel('Normalized weight')
ax.set_title('MAE-Based Weights / MAE 기반 가중치')

# Weighted vs simple mean
ax = axes[1]
ax.scatter(y_pred_simple, y_pred_weighted, alpha=0.3, s=10, c=y_test, cmap='RdBu_r')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_xlabel('Simple Mean Probability')
ax.set_ylabel('MAE-Weighted Mean Probability')
ax.set_title('Weighted vs Simple Mean / 가중 vs 단순 평균')

# Distribution of final forecasts
ax = axes[2]
ax.hist(y_pred_weighted[y_test == 0], bins=30, alpha=0.6, label='Non-storm', color='steelblue')
ax.hist(y_pred_weighted[y_test == 1], bins=30, alpha=0.6, label='Storm', color='coral')
ax.axvline(0.5, color='k', linestyle='--', label='Threshold 0.5')
ax.set_xlabel('Weighted Storm Probability')
ax.set_ylabel('Count')
ax.set_title('Forecast Distribution by True Class\n실제 클래스별 예보 분포')
ax.legend()

plt.tight_layout()
plt.show()

print(f"MAE range: {mae_vals.min():.1f} – {mae_vals.max():.1f} km/s")
print(f"Weight ratio (max/min): {w_norm.max()/w_norm.min():.1f}x")

## Part 5: Evaluation Metrics — ROC AUC & BSS / 평가 메트릭

논문에서 사용하는 두 가지 핵심 메트릭을 직접 구현합니다:
We implement the two key metrics used in the paper from scratch:

**ROC AUC** (Equations 6): 판별 능력 — threshold를 변화시키며 TPR vs FPR 커브를 그리고 면적을 계산.
Discriminative skill — plot TPR vs FPR by varying threshold and compute area.

**Brier Skill Score** (Equations 7-8): 확률적 정확도 — 확률 예보의 MSE를 기후학 대비 스킬로 변환.
Probabilistic accuracy — convert probability forecast MSE to skill relative to climatology.

$$BS = \frac{1}{N}\sum(p_i - o_i)^2, \quad BSS_{clim} = 1 - \frac{BS}{BS_{clim}}$$

In [ ]:
def compute_roc_curve(
    y_true: np.ndarray,
    y_scores: np.ndarray,
    n_thresholds: int = 200,
) -> Tuple[np.ndarray, np.ndarray, float]:
    """Compute ROC curve and AUC from scratch.

    Args:
        y_true: True binary labels (n_samples,).
        y_scores: Predicted probabilities (n_samples,).
        n_thresholds: Number of threshold values to evaluate.

    Returns:
        Tuple of (fpr_array, tpr_array, auc_value).
    """
    thresholds = np.linspace(1, 0, n_thresholds)
    tpr_list = []
    fpr_list = []

    for thresh in thresholds:
        y_pred = (y_scores >= thresh).astype(int)
        tp = np.sum((y_pred == 1) & (y_true == 1))
        fp = np.sum((y_pred == 1) & (y_true == 0))
        fn = np.sum((y_pred == 0) & (y_true == 1))
        tn = np.sum((y_pred == 0) & (y_true == 0))

        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        tpr_list.append(tpr)
        fpr_list.append(fpr)

    fpr_arr = np.array(fpr_list)
    tpr_arr = np.array(tpr_list)

    # AUC via trapezoidal rule
    auc = np.trapz(tpr_arr, fpr_arr)
    return fpr_arr, tpr_arr, auc


def compute_brier_score(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    """Compute Brier Score (Equation 7).

    Args:
        y_true: True binary labels (n_samples,).
        y_prob: Predicted probabilities (n_samples,).

    Returns:
        Brier Score value.
    """
    return np.mean((y_prob - y_true) ** 2)


def compute_bss_clim(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    """Compute BSS relative to climatology (Equation 8).

    For a balanced dataset, climatology forecast is p=0.5 for all,
    giving BS_clim = 0.25.

    Args:
        y_true: True binary labels (n_samples,).
        y_prob: Predicted probabilities (n_samples,).

    Returns:
        BSS_clim value.
    """
    bs = compute_brier_score(y_true, y_prob)
    bs_clim = compute_brier_score(y_true, np.full_like(y_true, 0.5, dtype=float))
    return 1 - bs / bs_clim


def compute_calibration(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    n_bins: int = 10,
) -> Tuple[np.ndarray, np.ndarray]:
    """Compute calibration curve (Figure 6 from the paper).

    Args:
        y_true: True binary labels.
        y_prob: Predicted probabilities.
        n_bins: Number of probability bins.

    Returns:
        Tuple of (mean_predicted, fraction_positive) per bin.
    """
    bin_edges = np.linspace(0, 1, n_bins + 1)
    mean_pred = []
    frac_pos = []

    for i in range(n_bins):
        mask = (y_prob >= bin_edges[i]) & (y_prob < bin_edges[i + 1])
        if mask.sum() > 0:
            mean_pred.append(y_prob[mask].mean())
            frac_pos.append(y_true[mask].mean())

    return np.array(mean_pred), np.array(frac_pos)


# --- Evaluate all methods ---
# Weighted mean (paper's approach)
fpr_w, tpr_w, auc_w = compute_roc_curve(y_test, y_pred_weighted)
bss_w = compute_bss_clim(y_test, y_pred_weighted)

# Simple mean (baseline)
fpr_s, tpr_s, auc_s = compute_roc_curve(y_test, y_pred_simple)
bss_s = compute_bss_clim(y_test, y_pred_simple)

# Persistence baseline (use last known label as prediction)
y_persist = np.roll(y_test, 1).astype(float)
y_persist[0] = 0.5
fpr_p, tpr_p, auc_p = compute_roc_curve(y_test, y_persist)
bss_p = compute_bss_clim(y_test, y_persist)

# --- Visualization (reproducing Figures 4 and 6 concepts) ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ROC curves
ax = axes[0]
ax.plot(fpr_w, tpr_w, 'b-', linewidth=2, label=f'Weighted Mean (AUC={auc_w:.3f})')
ax.plot(fpr_s, tpr_s, 'g--', linewidth=2, label=f'Simple Mean (AUC={auc_s:.3f})')
ax.plot(fpr_p, tpr_p, 'r:', linewidth=2, label=f'Persistence (AUC={auc_p:.3f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves / ROC 곡선')
ax.legend(loc='lower right')

# BSS comparison (bar chart)
ax = axes[1]
methods = ['Weighted\nMean', 'Simple\nMean', 'Persistence']
bss_values = [bss_w, bss_s, bss_p]
colors = ['steelblue', 'seagreen', 'coral']
bars = ax.bar(methods, bss_values, color=colors, edgecolor='black', linewidth=0.5)
ax.axhline(0.2, color='k', linestyle='--', alpha=0.5, label='Meaningful skill (0.2)')
ax.axhline(0, color='k', linewidth=0.5)
ax.set_ylabel('BSS_clim')
ax.set_title('Brier Skill Score / Brier 스킬 점수')
ax.legend()
for bar, val in zip(bars, bss_values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', fontsize=10)

# Calibration plot (Figure 6 concept)
ax = axes[2]
pred_bins, obs_freq = compute_calibration(y_test, y_pred_weighted)
ax.plot(pred_bins, obs_freq, 'bo-', linewidth=2, markersize=8, label='Weighted Mean')
pred_bins_s, obs_freq_s = compute_calibration(y_test, y_pred_simple)
ax.plot(pred_bins_s, obs_freq_s, 'gs--', linewidth=2, markersize=6, label='Simple Mean')
ax.plot([0, 1], [0, 1], 'r--', linewidth=1, label='Perfect calibration')
ax.set_xlabel('Predicted Probability')
ax.set_ylabel('Observed Frequency')
ax.set_title('Calibration Plot / 보정 그래프 (cf. Figure 6)')
ax.legend()

plt.tight_layout()
plt.show()

print("=== Performance Summary / 성능 요약 ===")
print(f"{'Method':<20} {'ROC AUC':>10} {'BSS_clim':>10} {'Brier Score':>12}")
print("-" * 55)
print(f"{'Weighted Mean':<20} {auc_w:>10.3f} {bss_w:>10.3f} {compute_brier_score(y_test, y_pred_weighted):>12.3f}")
print(f"{'Simple Mean':<20} {auc_s:>10.3f} {bss_s:>10.3f} {compute_brier_score(y_test, y_pred_simple):>12.3f}")
print(f"{'Persistence':<20} {auc_p:>10.3f} {bss_p:>10.3f} {compute_brier_score(y_test, y_persist):>12.3f}")
print(f"{'Climatology (0.5)':<20} {'0.500':>10} {'0.000':>10} {'0.250':>12}")

## Part 6: scikit-learn Comparison / scikit-learn 비교

직접 구현한 로지스틱 회귀와 scikit-learn의 `LogisticRegression`을 비교합니다.
논문에서도 scikit-learn의 L-BFGS 솔버를 사용했습니다.

Compare our from-scratch logistic regression with scikit-learn's `LogisticRegression`.
The paper also used scikit-learn's L-BFGS solver.

In [ ]:
from sklearn.linear_model import LogisticRegression as SklearnLR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, brier_score_loss

# Train sklearn classifiers per ensemble member (paper's actual approach)
sklearn_probs = np.zeros((n_ensemble_demo, len(y_test)))

for k in range(n_ensemble_demo):
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_trains[k])
    X_te = scaler.transform(X_tests[k])

    clf_sk = SklearnLR(solver='lbfgs', max_iter=1000)
    clf_sk.fit(X_tr, y_train)
    sklearn_probs[k] = clf_sk.predict_proba(X_te)[:, 1]

# MAE-weighted aggregation with sklearn classifiers
y_pred_sklearn = mae_weighted_mean(sklearn_probs, mae_vals)

# Evaluate
auc_sk = roc_auc_score(y_test, y_pred_sklearn)
bs_sk = brier_score_loss(y_test, y_pred_sklearn)
bss_sk = 1 - bs_sk / 0.25  # BS_clim = 0.25 for balanced dataset

# --- Comparison visualization ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: our implementation vs sklearn
ax = axes[0]
ax.scatter(y_pred_weighted, y_pred_sklearn, alpha=0.3, s=10,
           c=y_test, cmap='RdBu_r', edgecolors='none')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax.set_xlabel('From-Scratch Weighted Probability')
ax.set_ylabel('scikit-learn Weighted Probability')
ax.set_title('From-Scratch vs scikit-learn / 직접 구현 vs scikit-learn')
ax.set_aspect('equal')

# Bar comparison
ax = axes[1]
x = np.arange(2)
width = 0.35
bars1 = ax.bar(x - width / 2, [auc_w, bss_w], width, label='From Scratch', color='steelblue')
bars2 = ax.bar(x + width / 2, [auc_sk, bss_sk], width, label='scikit-learn', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(['ROC AUC', 'BSS_clim'])
ax.set_title('Performance Comparison / 성능 비교')
ax.legend()
for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print("=== From-Scratch vs scikit-learn Comparison ===")
print(f"{'Metric':<15} {'From-Scratch':>15} {'scikit-learn':>15} {'Difference':>12}")
print("-" * 60)
print(f"{'ROC AUC':<15} {auc_w:>15.4f} {auc_sk:>15.4f} {abs(auc_w-auc_sk):>12.4f}")
print(f"{'BSS_clim':<15} {bss_w:>15.4f} {bss_sk:>15.4f} {abs(bss_w-bss_sk):>12.4f}")
print(f"\nCorrelation between predictions: {np.corrcoef(y_pred_weighted, y_pred_sklearn)[0,1]:.4f}")

## Summary / 요약

### 구현된 알고리즘 / Implemented Algorithms

| Concept / 개념 | This Paper / 이 논문 | Our Implementation / 우리 구현 | Modern Equivalent / 현대 동등물 |
|---|---|---|---|
| 앙상블 생성 / Ensemble generation | MAS Carrington map + 사인파 섭동 (100 members) | 합성 Carrington map + 동일 섭동 기법 | HUXt Python package (실제 MAS 데이터 필요) |
| 태양풍 전파 / SW propagation | HUXt 1D reduced-physics model | Kinematic propagation (simplified) | `huxt` package on PyPI |
| 폭풍 분류 / Storm classification | Logistic regression per member (scikit-learn L-BFGS) | NumPy from scratch + scikit-learn 비교 | scikit-learn `LogisticRegression` |
| 앙상블 집약 / Ensemble aggregation | MAE-weighted mean (Eq. 3-5) | NumPy from scratch (동일) | Custom (논문 고유 방법) |
| ROC AUC | Standard metric | NumPy from scratch | `sklearn.metrics.roc_auc_score` |
| BSS_clim | Brier Skill Score vs climatology | NumPy from scratch | `sklearn.metrics.brier_score_loss` |
| Calibration | Reliability diagram (10-bin) | NumPy from scratch | `sklearn.calibration.calibration_curve` |

### 논문의 실제 데이터 vs 우리의 합성 데이터 / Paper's Real Data vs Our Synthetic Data

| Aspect | Paper | This Notebook |
|---|---|---|
| Carrington maps | MAS 3D MHD (1995–2024) | Synthetic with realistic structure |
| Propagation | HUXt (open-source) | Simplified kinematic model |
| OMNI data | Real hourly observations | Ensemble mean + Gaussian noise |
| Ensemble size | 100 members | 20–50 members |
| Storms | 2345 balanced (Hp30 ≥ 4.66) | ~500 balanced (synthetic) |

### 다음 논문과의 연결 / Connection to Next Papers

| Paper | Connection / 연결 |
|---|---|
| **SW #38** Abduallah et al. (2024) — SYMHnet | GNN+BiLSTM으로 SYM-H 예측. 본 논문의 로지스틱 회귀보다 복잡한 딥러닝 접근법 / Deep learning approach more complex than this paper's logistic regression |
| **SW #39** Wang et al. (2026) — SINet | F10.7/F30 예측에 TimesNet 아키텍처 사용. 태양 활동 지수 예측이라는 다른 타깃이지만 동일한 ML+physics 패러다임 / Different target (solar indices) but same ML+physics paradigm |
| **SW #33** Camporeale et al. (2019) | 우주기상 ML 리뷰. 본 논문이 이 리뷰에서 제시한 모범 사례를 따름 / ML in space weather review. This paper follows best practices outlined there |
| **SW #36** Owens et al. (2021) | 극한 우주기상 사건의 태양 주기 의존성. 본 논문의 확률적 예보가 이러한 극한 사건 대비에 기여 / Extreme event solar cycle dependence. This paper's probabilistic forecasts contribute to preparedness |